<a href="#">
<center><img src="https://th.bing.com/th/id/OIG2.xEMFKi2pyFcX8UuRkLQq?pid=ImgGn"/></center>
</a>

# Follow the Steps below to Train YOLOv8 Object Detection on a Custom Dataset

---


Ultralytics YOLOv8 is one of the latest version of the YOLO (You Only Look Once) object detection and image segmentation model developed by Ultralytics. The YOLOv8 model is designed to be fast, accurate, and easy to use, making it an excellent choice for a wide range of object detection and image segmentation tasks. It can be trained on large datasets and is capable of running on a variety of hardware platforms, from CPUs to GPUs.

## Pro Tip: Use GPU Acceleration

If you are running this notebook in Google Colab, navigate to `Edit` -> `Notebook settings` -> `Hardware accelerator`, set it to `GPU`, and then click `Save`. This will ensure your notebook uses a GPU, which will significantly speed up model training times.

## Steps in this Tutorial

- Install YOLOv8
- (OPTIONAL) Install Weights & Biases for experiment tracking
- Custom Training
- Validate Custom Model
- Inference with Custom Model


## Before you start

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Edit` -> `Notebook settings` -> `Hardware accelerator`, set it to `GPU`, and then click `Save`.

In [ ]:
!nvidia-smi

## Install YOLOv8

YOLOv8 can be installed in two ways - from the source and via pip. The recommended method is via pip as shown below.

In [ ]:
# Pip install method (recommended)

# !pip install --upgrade ultralytics
# !pip install ultralytics==8.0.134
# !pip install ultralytics==8.0.120
!pip install ultralytics


from IPython import display
display.clear_output()

import ultralytics
ultralytics.checks()

In [ ]:
from ultralytics import YOLO

from IPython.display import display, Image

### Install Weights & Biases for Experiment Tracking (optional)

In [ ]:
!pip install wandb -qU

In [ ]:
# Log in to your W&B account
import wandb
wandb.login()

## CLI Basics

If you want to train, validate or run inference on models and don't need to make any modifications to the code, using YOLO command line interface is the easiest way to get started. Read more about CLI in [Ultralytics YOLO Docs](https://docs.ultralytics.com/usage/cli/).

```
yolo task=detect    mode=train    model=yolov8n.yaml      args...
          classify       predict        yolov8n-cls.yaml  args...
          segment        val            yolov8n-seg.yaml  args...
                         export         yolov8n.pt        format=onnx  args...
```

## Prepare the Training Environment

In [ ]:
# Prepare the environment by connecting to your Google Drive
# We will load a custom dataset from our Google Drive Account

import os
from google.colab import drive

drive.mount('/content/drive')

HOME = "/content/drive/MyDrive/ROBOKEN-2024/Machine-Vision/Object-Detection-Model"
print(HOME)

## Running the cell below will start training the model.


In [ ]:
%cd {HOME}

import os


SAVE_DIR = 'roboken-object-detection' # Model weights will be stored here
DATASET_PATH = './datasets/roboken-dataset-v3/data.yaml' # Path to the data.yaml file
PATH_TO_LAST = "./roboken-object-detection/yolov8-object-detection-v1-0/weights/last.pt" # [TRANSFER LEARNING] Path to the last.pt file in the weights folder
NAME = 'yolov8-object-detection-v1-0' # Give this run a suitable name
EPOCHS = 1000 # Specify the number of epochs for training

os.makedirs(SAVE_DIR, exist_ok=True)


!yolo task=detect mode=train model=yolov8n.pt data={DATASET_PATH} project={SAVE_DIR} name={NAME} epochs={EPOCHS} save_period=15 imgsz=640 plots=True patience=100

## Resuming Training from the last epoch

In case the training process was interrupted in the first run, execute the cell below to continue training from the last epoch.

Re-running the cell above will start the training process from epoch 0!

In [ ]:
%cd {HOME}

import os


SAVE_DIR = 'roboken-object-detection' # Model weights will be stored here
DATASET_PATH = './datasets/roboken-dataset-v3/data.yaml' # Path to the data.yaml file
PATH_TO_LAST = "./roboken-object-detection/yolov8-object-detection-v1-0/weights/last.pt" # [TRANSFER LEARNING] Path to the last.pt file in the weights folder
NAME = 'yolov8-object-detection-v1-0' # Give this run a suitable name
EPOCHS = 1000 # Specify the number of epochs for training

os.makedirs(SAVE_DIR, exist_ok=True)


!yolo task=detect mode=train model={PATH_TO_LAST} data={DATASET_PATH} project={SAVE_DIR} name={NAME} epochs={EPOCHS} save_period=15 imgsz=640 plots=True patience=100

## Validate Custom Model

In [ ]:
RESULTS_DIR = "/content/drive/MyDrive/ROBOKEN-2024/Machine-Vision/Object-Detection-Model/"

%cd {RESULTS_DIR}

!yolo task=detect mode=val model=./yolov8-object-detection-v1-0/weights/best.pt data={HOME}/datasets/roboken-dataset-v3/data.yaml

## Inference with Custom Model

In [ ]:
RESULTS_DIR = "/content/drive/MyDrive/ROBOKEN-2024/Machine-Vision/Object-Detection-Model/"

%cd {RESULTS_DIR}

!yolo task=detect mode=predict model=./yolov8-object-detection-v1-0/weights/best.pt conf=0.25 source=../datasets/roboken-dataset-v3/test/images save=True

## Convert Model to ONNX


In [ ]:
%cd {HOME}

# Load a model
model = YOLO('yolov8n.pt')  # load an official model
model = YOLO('Object-Detection-Model/yolov8-object-detection-v1-0/weights/best.pt')  # load a custom trained model

# Export the model
model.export(format='onnx')

---

## 🏆 Deploy model on Roboflow

Once you have finished training your YOLOv8 model, you’ll have a set of trained weights ready for use. These weights will be in the `/runs/detect/train/weights/best.pt` folder of your project. You can upload your model weights to Roboflow Deploy to use your trained weights on our infinitely scalable infrastructure.

The `.deploy()` function in the [Roboflow pip package](https://docs.roboflow.com/python) now supports uploading YOLOv8 weights.

To upload model weights, add the following code to the “Inference with Custom Model” section in the aforementioned notebook:

In [ ]:
!pip install roboflow

In [ ]:
!pip install --upgrade ultralytics

In [ ]:
RESULTS_DIR = "/content/drive/MyDrive/ROBOKEN-2024/Machine-Vision/Object-Detection-Model/"
%cd {RESULTS_DIR}

import roboflow

roboflow.login()

rf = roboflow.Roboflow()

# # create a project
# rf.create_project(
#     project_name="project name",
#     project_type="project-type",
#     license="project-license" # "private" for private projects
# )

workspace = rf.workspace("roboken-2024")
project = workspace.project("roboken-object-detection")
version = project.version("1")


# upload model weights
version.deploy(model_type="yolov8", model_path="./yolov8-object-detection-v1-0/")